# ⚾ MLB Analytics Pipeline
**Run cells top to bottom. Every cell is self-contained — no external files needed.**

| Cell | Purpose |
|------|---------|
| 1 | Install packages |
| 2 | Mount Google Drive (optional but recommended) |
| 3 | All imports |
| 4 | Database setup |
| 5 | Park factors table |
| 6 | Fetch today's schedule |
| 7 | Fetch batting stats |
| 8 | Fetch pitching stats |
| 9 | Fetch Statcast pitch data |
| 10 | **RUN THIS DAILY** — nightly update |
| 11 | Today's matchup preview |
| 12 | Batter vs pitcher breakdown |
| 13 | Database summary |


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Install packages  (run once, skip after that)
# ═══════════════════════════════════════════════════════════
import subprocess, sys

packages = [
    'pybaseball',
    'mlb-statsapi',
    'pandas',
    'numpy',
    'requests',
    'sqlalchemy',
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✓' if result.returncode == 0 else '✗'
    print(f'  {status} {pkg}')

print('\nDone. Restart runtime if this is your first time (Runtime → Restart session), then continue from Cell 2.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Mount Google Drive  (keeps your DB between sessions)
# Set MOUNT_DRIVE = False to skip and use temporary storage instead
# ═══════════════════════════════════════════════════════════
MOUNT_DRIVE = True

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs('/content/drive/MyDrive/mlb_analytics', exist_ok=True)
    DB_PATH = '/content/drive/MyDrive/mlb_analytics/mlb.db'
    print(f'✓ Drive mounted. DB → {DB_PATH}')
else:
    DB_PATH = '/content/mlb.db'
    print(f'Using temporary storage. DB → {DB_PATH}')
    print('Data will be lost when the session ends.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Imports
# ═══════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import sqlite3
import os
import time
import warnings
from datetime import datetime, date, timedelta

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.3f}'.format)

# ── statsapi (MLB schedule, rosters, probable pitchers) ──────────────────────
try:
    import statsapi
    print('✓ statsapi')
except ImportError:
    print('✗ statsapi not installed — run Cell 1 again')

# ── pybaseball (only stable exports used) ────────────────────────────────────
try:
    from pybaseball import statcast, batting_stats, pitching_stats, playerid_lookup, cache
    cache.enable()
    print('✓ pybaseball (statcast, batting_stats, pitching_stats, playerid_lookup)')
except ImportError as e:
    print(f'✗ pybaseball import error: {e}')
    print('  Run Cell 1 again, then restart runtime')

SEASON = datetime.now().year
print(f'\nReady. Season: {SEASON}')
print(f'DB: {DB_PATH}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Database setup
# ═══════════════════════════════════════════════════════════
def get_conn():
    return sqlite3.connect(DB_PATH)

def init_db():
    conn = get_conn()
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS schedule (
            game_id         TEXT PRIMARY KEY,
            game_date       TEXT,
            away_team       TEXT,
            home_team       TEXT,
            venue_name      TEXT,
            game_time       TEXT,
            away_pitcher    TEXT,
            home_pitcher    TEXT,
            away_pitcher_id TEXT,
            home_pitcher_id TEXT,
            fetched_at      TEXT
        );

        CREATE TABLE IF NOT EXISTS batting_stats (
            player_id    TEXT,
            name         TEXT,
            team         TEXT,
            season       INTEGER,
            pa           REAL, ab REAL, h REAL, hr REAL, rbi REAL,
            avg          REAL, obp REAL, slg REAL, ops REAL,
            woba         REAL, xwoba REAL, xba REAL, xslg REAL,
            barrel_pct   REAL, hard_hit_pct REAL,
            k_pct        REAL, bb_pct REAL,
            exit_velo    REAL, launch_angle REAL,
            sprint_speed REAL,
            fetched_at   TEXT,
            PRIMARY KEY (player_id, season)
        );

        CREATE TABLE IF NOT EXISTS pitching_stats (
            player_id   TEXT,
            name        TEXT,
            team        TEXT,
            season      INTEGER,
            ip          REAL, era REAL, fip REAL, xfip REAL, siera REAL,
            k_pct       REAL, bb_pct REAL, k_bb_pct REAL,
            hr9         REAL, whip REAL,
            gb_pct      REAL, fb_pct REAL, hard_pct REAL,
            stuff_plus  REAL, loc_plus REAL,
            fetched_at  TEXT,
            PRIMARY KEY (player_id, season)
        );

        CREATE TABLE IF NOT EXISTS statcast_pitches (
            id                  INTEGER PRIMARY KEY AUTOINCREMENT,
            game_date           TEXT,
            game_pk             INTEGER,
            pitcher_id          INTEGER,
            batter_id           INTEGER,
            pitch_type          TEXT,
            pitch_name          TEXT,
            velo                REAL,
            spin_rate           REAL,
            pfx_x               REAL,
            pfx_z               REAL,
            plate_x             REAL,
            plate_z             REAL,
            zone                INTEGER,
            exit_velo           REAL,
            launch_angle        REAL,
            hit_distance        REAL,
            xwoba               REAL,
            events              TEXT,
            description         TEXT,
            batter_hand         TEXT,
            pitcher_hand        TEXT,
            home_team           TEXT,
            away_team           TEXT,
            bb_type             TEXT,
            barrel              INTEGER,
            fetched_at          TEXT
        );
    """)
    conn.commit()
    conn.close()
    print(f'✓ Database initialised at {DB_PATH}')

init_db()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Park factors  (hardcoded 3-year averages, all 30 parks)
# 100 = neutral  |  >100 = hitter-friendly  |  <100 = pitcher-friendly
# HR factor, runs factor, hits factor all included
# ═══════════════════════════════════════════════════════════

PARK_FACTORS = pd.DataFrame([
    # team_abbr, team_name, venue, basic, hr_factor, runs_factor, h_factor, altitude_ft
    ('COL', 'Colorado Rockies',         'Coors Field',                  115, 130, 119, 112, 5183),
    ('CIN', 'Cincinnati Reds',          'Great American Ball Park',     112, 122, 111, 109,  550),
    ('TEX', 'Texas Rangers',            'Globe Life Field',             108, 115, 109, 107,  551),
    ('BOS', 'Boston Red Sox',           'Fenway Park',                  107, 104, 107, 111,   20),
    ('PHI', 'Philadelphia Phillies',    'Citizens Bank Park',           107, 114, 107, 106,   20),
    ('CHC', 'Chicago Cubs',             'Wrigley Field',                106, 108, 105, 107,  595),
    ('ATL', 'Atlanta Braves',           'Truist Park',                  104, 108, 103, 103, 1050),
    ('MIL', 'Milwaukee Brewers',        'American Family Field',        103, 105, 103, 102,  635),
    ('TOR', 'Toronto Blue Jays',        'Rogers Centre',                102, 103, 101, 101,  300),
    ('HOU', 'Houston Astros',           'Minute Maid Park',             101, 100, 101, 101,   43),
    ('NYY', 'New York Yankees',         'Yankee Stadium',               101, 106, 101, 100,   55),
    ('ARI', 'Arizona Diamondbacks',     'Chase Field',                  100, 100, 100, 100, 1100),
    ('STL', 'St. Louis Cardinals',      'Busch Stadium',                 99,  97,  99,  99,  466),
    ('BAL', 'Baltimore Orioles',        'Camden Yards',                  99, 100,  99, 100,   20),
    ('KC',  'Kansas City Royals',       'Kauffman Stadium',              98,  95,  98,  99,  750),
    ('LAD', 'Los Angeles Dodgers',      'Dodger Stadium',                97,  97,  98,  97,  515),
    ('CLE', 'Cleveland Guardians',      'Progressive Field',             97,  93,  97,  97,  653),
    ('DET', 'Detroit Tigers',           'Comerica Park',                 96,  90,  96,  97,  600),
    ('CWS', 'Chicago White Sox',        'Guaranteed Rate Field',         96,  96,  96,  96,  595),
    ('PIT', 'Pittsburgh Pirates',       'PNC Park',                      95,  88,  95,  96,  730),
    ('SF',  'San Francisco Giants',     'Oracle Park',                   94,  85,  93,  95,   52),
    ('MIN', 'Minnesota Twins',          'Target Field',                  94,  91,  93,  95,  830),
    ('TB',  'Tampa Bay Rays',           'Tropicana Field',               93,  89,  93,  94,   15),
    ('OAK', 'Oakland Athletics',        'Oakland Coliseum',              93,  88,  92,  94,   42),
    ('SEA', 'Seattle Mariners',         'T-Mobile Park',                 93,  87,  92,  94,   17),
    ('LAA', 'Los Angeles Angels',       'Angel Stadium',                 92,  91,  92,  93,  160),
    ('SD',  'San Diego Padres',         'Petco Park',                    91,  85,  91,  92,   20),
    ('MIA', 'Miami Marlins',            'loanDepot park',                90,  83,  89,  91,    6),
    ('NYM', 'New York Mets',            'Citi Field',                    89,  83,  88,  90,   20),
    ('WSH', 'Washington Nationals',     'Nationals Park',                88,  82,  87,  89,   25),
], columns=['abbr', 'team_name', 'venue', 'basic', 'hr_factor', 'runs_factor', 'h_factor', 'altitude_ft'])

def get_park_factor(team_name_or_venue: str) -> dict:
    """
    Look up park factors by full team name, abbreviation, or venue name.
    Returns dict with basic, hr_factor, runs_factor, h_factor, altitude_ft.
    Returns neutral park (100) if not found.
    """
    q = team_name_or_venue.lower()
    mask = (
        PARK_FACTORS['team_name'].str.lower().str.contains(q) |
        PARK_FACTORS['abbr'].str.lower().str.contains(q) |
        PARK_FACTORS['venue'].str.lower().str.contains(q)
    )
    if mask.any():
        return PARK_FACTORS[mask].iloc[0].to_dict()
    return {'team_name': 'Unknown', 'venue': team_name_or_venue,
            'basic': 100, 'hr_factor': 100, 'runs_factor': 100,
            'h_factor': 100, 'altitude_ft': 0}

print('✓ Park factors loaded for all 30 parks')
print()
print('Most hitter-friendly (by HR factor):')
print(PARK_FACTORS.nlargest(5, 'hr_factor')[['team_name','venue','hr_factor','runs_factor']].to_string(index=False))
print()
print('Most pitcher-friendly (by HR factor):')
print(PARK_FACTORS.nsmallest(5, 'hr_factor')[['team_name','venue','hr_factor','runs_factor']].to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Fetch today's schedule + probable pitchers
# ═══════════════════════════════════════════════════════════

def fetch_schedule(target_date: str = None) -> pd.DataFrame:
    """
    Pulls today's MLB schedule with probable starters and venue.
    Args:
        target_date: 'YYYY-MM-DD'. Defaults to today.
    """
    target_date = target_date or date.today().strftime('%Y-%m-%d')
    print(f'Fetching schedule for {target_date}...')

    try:
        raw = statsapi.schedule(date=target_date)
    except Exception as e:
        print(f'  ✗ Error: {e}')
        return pd.DataFrame()

    if not raw:
        print(f'  No games scheduled for {target_date}.')
        return pd.DataFrame()

    rows = []
    for g in raw:
        rows.append({
            'game_id':         str(g.get('game_id', '')),
            'game_date':       target_date,
            'away_team':       g.get('away_name', ''),
            'home_team':       g.get('home_name', ''),
            'venue_name':      g.get('venue_name', ''),
            'game_time':       g.get('game_datetime', ''),
            'away_pitcher':    g.get('away_probable_pitcher', 'TBD'),
            'home_pitcher':    g.get('home_probable_pitcher', 'TBD'),
            'away_pitcher_id': str(g.get('away_probable_pitcher_id', '')),
            'home_pitcher_id': str(g.get('home_probable_pitcher_id', '')),
            'fetched_at':      datetime.now().isoformat(),
        })

    df = pd.DataFrame(rows)

    conn = get_conn()
    df.to_sql('schedule', conn, if_exists='replace', index=False)
    conn.close()

    print(f'  ✓ {len(df)} games saved')
    print()
    for _, r in df.iterrows():
        pf = get_park_factor(r['home_team'])
        print(f"  {r['away_team']:28s} @ {r['home_team']:28s}")
        print(f"  Venue: {r['venue_name']:30s}  HR factor: {pf['hr_factor']}  Runs factor: {pf['runs_factor']}")
        print(f"  Away SP: {r['away_pitcher']:25s}  Home SP: {r['home_pitcher']}")
        print()

    return df

schedule_df = fetch_schedule()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Fetch season batting stats from FanGraphs
# Includes xwOBA, Barrel%, Hard Hit%, K%, BB% via pybaseball
# ═══════════════════════════════════════════════════════════

# Column map: FanGraphs name → our clean name
BAT_COLS = {
    'IDfg': 'player_id', 'Name': 'name', 'Team': 'team',
    'PA': 'pa', 'AB': 'ab', 'H': 'h', 'HR': 'hr', 'RBI': 'rbi',
    'AVG': 'avg', 'OBP': 'obp', 'SLG': 'slg', 'OPS': 'ops',
    'wOBA': 'woba', 'xwOBA': 'xwoba', 'xBA': 'xba', 'xSLG': 'xslg',
    'Barrel%': 'barrel_pct', 'HardHit%': 'hard_hit_pct',
    'K%': 'k_pct', 'BB%': 'bb_pct',
    'EV': 'exit_velo', 'LA': 'launch_angle',
    'Sprint Speed': 'sprint_speed',
}

def fetch_batting_stats(season: int = None, min_pa: int = 50) -> pd.DataFrame:
    season = season or SEASON
    print(f'Fetching {season} batting stats (min {min_pa} PA)...')
    try:
        df = batting_stats(season, qual=min_pa)
    except Exception as e:
        print(f'  ✗ Error: {e}')
        return pd.DataFrame()

    # Keep only columns that exist in the response
    keep = {k: v for k, v in BAT_COLS.items() if k in df.columns}
    df = df[list(keep.keys())].rename(columns=keep).copy()
    df['season'] = season
    df['fetched_at'] = datetime.now().isoformat()
    df['player_id'] = df['player_id'].astype(str)

    # Normalise percentage columns: FanGraphs sometimes returns 0-100, sometimes 0-1
    pct_cols = ['k_pct', 'bb_pct', 'barrel_pct', 'hard_hit_pct']
    for col in pct_cols:
        if col in df.columns:
            if df[col].dropna().median() > 1:
                df[col] = df[col] / 100

    conn = get_conn()
    df.to_sql('batting_stats', conn, if_exists='replace', index=False)
    conn.close()

    print(f'  ✓ {len(df)} batters saved')
    print()
    show_cols = [c for c in ['name','team','pa','avg','obp','slg','xwoba','barrel_pct','k_pct'] if c in df.columns]
    print('Top 10 by xwOBA:')
    print(df.nlargest(10, 'xwoba')[show_cols].to_string(index=False))
    return df

batting_df = fetch_batting_stats()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Fetch season pitching stats from FanGraphs
# Includes FIP, xFIP, SIERA, K%, Stuff+ via pybaseball
# ═══════════════════════════════════════════════════════════

PITCH_COLS = {
    'IDfg': 'player_id', 'Name': 'name', 'Team': 'team',
    'IP': 'ip', 'ERA': 'era', 'FIP': 'fip', 'xFIP': 'xfip', 'SIERA': 'siera',
    'K%': 'k_pct', 'BB%': 'bb_pct', 'K-BB%': 'k_bb_pct',
    'HR/9': 'hr9', 'WHIP': 'whip',
    'GB%': 'gb_pct', 'FB%': 'fb_pct', 'Hard%': 'hard_pct',
    'Stuff+': 'stuff_plus', 'Location+': 'loc_plus',
}

def fetch_pitching_stats(season: int = None, min_ip: int = 20) -> pd.DataFrame:
    season = season or SEASON
    print(f'Fetching {season} pitching stats (min {min_ip} IP)...')
    try:
        df = pitching_stats(season, qual=min_ip)
    except Exception as e:
        print(f'  ✗ Error: {e}')
        return pd.DataFrame()

    keep = {k: v for k, v in PITCH_COLS.items() if k in df.columns}
    df = df[list(keep.keys())].rename(columns=keep).copy()
    df['season'] = season
    df['fetched_at'] = datetime.now().isoformat()
    df['player_id'] = df['player_id'].astype(str)

    pct_cols = ['k_pct', 'bb_pct', 'k_bb_pct', 'gb_pct', 'fb_pct', 'hard_pct']
    for col in pct_cols:
        if col in df.columns:
            if df[col].dropna().median() > 1:
                df[col] = df[col] / 100

    conn = get_conn()
    df.to_sql('pitching_stats', conn, if_exists='replace', index=False)
    conn.close()

    print(f'  ✓ {len(df)} pitchers saved')
    print()
    show_cols = [c for c in ['name','team','ip','era','fip','xfip','k_pct','k_bb_pct','stuff_plus'] if c in df.columns]
    qualified = df[df['ip'] >= 50] if 'ip' in df.columns else df
    print('Top 10 qualified SPs by FIP:')
    print(qualified.nsmallest(10, 'fip')[show_cols].to_string(index=False))
    return df

pitching_df = fetch_pitching_stats()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Fetch Statcast pitch-level data
# Every pitch from Baseball Savant: velocity, spin, movement, outcome
#
# FIRST RUN: use days_back=30  (~2-5 min, ~500k rows)
# DAILY:     use nightly_update() in Cell 10 instead
# ═══════════════════════════════════════════════════════════

STATCAST_KEEP = [
    'game_date', 'game_pk', 'pitcher', 'batter',
    'pitch_type', 'pitch_name',
    'release_speed', 'release_spin_rate',
    'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'zone',
    'launch_speed', 'launch_angle', 'hit_distance_sc',
    'estimated_woba_using_speedangle',
    'events', 'description',
    'stand', 'p_throws',
    'home_team', 'away_team',
    'bb_type', 'barrel',
]

STATCAST_RENAME = {
    'pitcher':                              'pitcher_id',
    'batter':                               'batter_id',
    'release_speed':                        'velo',
    'release_spin_rate':                    'spin_rate',
    'launch_speed':                         'exit_velo',
    'hit_distance_sc':                      'hit_distance',
    'estimated_woba_using_speedangle':      'xwoba',
    'stand':                                'batter_hand',
    'p_throws':                             'pitcher_hand',
}

def fetch_statcast_data(days_back: int = 30, append: bool = False) -> pd.DataFrame:
    """
    Fetch pitch-level Statcast data.
    Args:
        days_back: How many days of data to pull. 30 = fast, 90 = richer.
        append:    True = add to existing data. False = replace.
    """
    end_dt   = date.today()
    start_dt = end_dt - timedelta(days=days_back)

    print(f'Fetching Statcast data: {start_dt} → {end_dt}  ({days_back} days)')
    print('  This takes 2–5 min for 30 days. Be patient, do not close the tab.')

    try:
        df = statcast(
            start_dt=start_dt.strftime('%Y-%m-%d'),
            end_dt=end_dt.strftime('%Y-%m-%d'),
        )
    except Exception as e:
        print(f'  ✗ Error: {e}')
        return pd.DataFrame()

    if df is None or df.empty:
        print('  No data returned (season may not have started yet).')
        return pd.DataFrame()

    # Keep only the columns we need (ignore missing ones gracefully)
    keep = [c for c in STATCAST_KEEP if c in df.columns]
    df = df[keep].rename(columns={k: v for k, v in STATCAST_RENAME.items() if k in keep}).copy()

    # Drop rows with no pitch type (warm-up pitches, data gaps)
    df = df.dropna(subset=['pitch_type'])
    df['fetched_at'] = datetime.now().isoformat()

    conn = get_conn()
    mode = 'append' if append else 'replace'
    df.to_sql('statcast_pitches', conn, if_exists=mode, index=False)
    conn.close()

    print(f'  ✓ {len(df):,} pitches saved (mode: {mode})')
    print()

    if 'pitch_type' in df.columns:
        counts = df['pitch_type'].value_counts().head(10)
        total  = len(df)
        print('  Pitch type breakdown:')
        for pt, n in counts.items():
            bar = '█' * int(n / total * 30)
            print(f'    {pt:5s}  {bar:<30s}  {n:>7,}  ({n/total:.1%})')

    return df

# FIRST RUN — pull 30 days of pitch data
# Increase days_back to 90 for a richer dataset (takes ~10 min)
statcast_df = fetch_statcast_data(days_back=30, append=False)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — RUN THIS EVERY MORNING
# Refreshes schedule + appends last 2 days of Statcast (~2 min)
# ═══════════════════════════════════════════════════════════

def nightly_update():
    print('=' * 55)
    print(f'  Nightly update — {date.today()}')
    print('=' * 55)
    fetch_schedule()
    fetch_statcast_data(days_back=2, append=True)
    print('Done. Run Cell 11 to see today\'s matchups.')

# Uncomment this line to run the daily update:
# nightly_update()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — Today's matchup preview
# Schedule + probable starters + FIP + park factors combined
# ═══════════════════════════════════════════════════════════

def matchup_preview() -> pd.DataFrame:
    conn = get_conn()

    try:
        sched = pd.read_sql('SELECT * FROM schedule', conn)
        pitch = pd.read_sql('SELECT * FROM pitching_stats', conn)
    except Exception as e:
        print(f'Error: {e}. Run Cells 6-8 first.')
        conn.close()
        return pd.DataFrame()
    conn.close()

    if sched.empty:
        print('No schedule data. Run Cell 6 first.')
        return pd.DataFrame()

    def get_sp_stats(name):
        if not name or name == 'TBD' or pitch.empty:
            return {}
        last = name.split()[-1].lower()
        mask = pitch['name'].str.lower().str.contains(last, na=False)
        if mask.any():
            return pitch[mask].iloc[0].to_dict()
        return {}

    def fmt(val, fmt_str='.2f'):
        if val and not (isinstance(val, float) and np.isnan(val)):
            return format(float(val), fmt_str)
        return 'N/A'

    print('═' * 65)
    print(f'  TODAY\'S MATCHUPS  —  {date.today()}')
    print('═' * 65)

    rows = []
    for _, g in sched.iterrows():
        pf  = get_park_factor(g['home_team'])
        asp = get_sp_stats(g['away_pitcher'])
        hsp = get_sp_stats(g['home_pitcher'])

        print(f"\n  {g['away_team']:28s} @ {g['home_team']}")
        print(f"  {g['venue_name']:40s}  HR factor: {pf['hr_factor']}  Altitude: {pf['altitude_ft']}ft")
        print(f"  Away SP: {g['away_pitcher']:28s}  FIP: {fmt(asp.get('fip'))}  xFIP: {fmt(asp.get('xfip'))}  K%: {fmt(asp.get('k_pct'),'.1%')}  Stuff+: {fmt(asp.get('stuff_plus'),'g')}")
        print(f"  Home SP: {g['home_pitcher']:28s}  FIP: {fmt(hsp.get('fip'))}  xFIP: {fmt(hsp.get('xfip'))}  K%: {fmt(hsp.get('k_pct'),'.1%')}  Stuff+: {fmt(hsp.get('stuff_plus'),'g')}")

        rows.append({
            'away': g['away_team'], 'home': g['home_team'],
            'venue': g['venue_name'],
            'hr_factor': pf['hr_factor'], 'runs_factor': pf['runs_factor'],
            'away_sp': g['away_pitcher'], 'away_fip': asp.get('fip'), 'away_xfip': asp.get('xfip'),
            'home_sp': g['home_pitcher'], 'home_fip': hsp.get('fip'), 'home_xfip': hsp.get('xfip'),
        })

    print('\n' + '═' * 65)
    return pd.DataFrame(rows)

today_df = matchup_preview()
today_df

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12 — Batter vs pitcher pitch-type breakdown
# Shows xwOBA, whiff rate, barrel rate by pitch type
# Change the names below to whoever you want to analyse
# ═══════════════════════════════════════════════════════════

def batter_vs_pitcher(batter_name: str, pitcher_name: str = None) -> pd.DataFrame:
    """
    Analyse how a batter performs against each pitch type.
    Optionally filter to a specific pitcher.

    Uses Statcast pitch-level data to show:
      - Average velocity and spin rate
      - Swing rate, whiff rate, barrel rate
      - xwOBA (quality of contact)
    """
    conn = get_conn()
    try:
        df = pd.read_sql('SELECT * FROM statcast_pitches', conn)
    except Exception as e:
        print(f'Error: {e}. Run Cell 9 first.')
        conn.close()
        return pd.DataFrame()
    conn.close()

    if df.empty:
        print('No Statcast data yet. Run Cell 9 first.')
        return pd.DataFrame()

    # Look up batter MLBAM ID
    try:
        parts = batter_name.strip().split()
        lookup = playerid_lookup(parts[-1], parts[0])
        if lookup.empty:
            print(f'Could not find "{batter_name}" in player database.')
            print('Try: last name only, or check spelling.')
            return pd.DataFrame()
        batter_id = int(lookup.iloc[0]['key_mlbam'])
    except Exception as e:
        print(f'Player lookup error: {e}')
        return pd.DataFrame()

    bdf = df[df['batter_id'] == batter_id].copy()

    # Optionally filter to specific pitcher
    if pitcher_name:
        try:
            pparts = pitcher_name.strip().split()
            plookup = playerid_lookup(pparts[-1], pparts[0])
            if not plookup.empty:
                pid = int(plookup.iloc[0]['key_mlbam'])
                bdf = bdf[bdf['pitcher_id'] == pid]
        except Exception:
            pass

    if bdf.empty:
        print(f'No pitches found for {batter_name} in current Statcast window.')
        print('Try fetching more data: fetch_statcast_data(days_back=60)')
        return pd.DataFrame()

    # Compute outcome flags
    bdf['swing']  = bdf['description'].str.contains('swinging|foul|hit_into_play', na=False)
    bdf['whiff']  = bdf['description'].str.contains('swinging_strike', na=False)
    bdf['barrel'] = bdf['barrel'] == 1

    summary = (
        bdf.groupby('pitch_type')
        .agg(
            count      = ('pitch_type',   'count'),
            avg_velo   = ('velo',          'mean'),
            avg_spin   = ('spin_rate',     'mean'),
            xwoba      = ('xwoba',         'mean'),
            swing_rate = ('swing',         'mean'),
            whiff_rate = ('whiff',         'mean'),
            barrel_rate= ('barrel',        'mean'),
        )
        .reset_index()
        .query('count >= 5')
        .sort_values('xwoba', ascending=False)
    )

    for col in ['avg_velo', 'avg_spin']:
        if col in summary.columns:
            summary[col] = summary[col].round(1)
    for col in ['xwoba', 'swing_rate', 'whiff_rate', 'barrel_rate']:
        if col in summary.columns:
            summary[col] = summary[col].round(3)

    label = f'{batter_name}' + (f' vs {pitcher_name}' if pitcher_name else ' — all pitchers')
    print(f'\n{label}')
    print(f'Total pitches in sample: {len(bdf):,}')
    print()
    print(summary.to_string(index=False))

    if not summary.empty:
        worst = summary.nsmallest(1, 'xwoba').iloc[0]
        best  = summary.nlargest(1,  'xwoba').iloc[0]
        print(f'\n  Best pitch to see:  {best["pitch_type"]:5s}  xwOBA {best["xwoba"]:.3f}')
        print(f'  Most vulnerable to: {worst["pitch_type"]:5s}  xwOBA {worst["xwoba"]:.3f}  Whiff rate {worst["whiff_rate"]:.1%}')

    return summary

# ── Change names below to any MLB player ────────────────────────────────────
BATTER  = 'Aaron Judge'
PITCHER = 'Gerrit Cole'   # Set to None to see vs all pitchers

result = batter_vs_pitcher(BATTER, PITCHER)
result

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 13 — Database summary
# Shows what's in each table so you know the pipeline worked
# ═══════════════════════════════════════════════════════════

conn = get_conn()
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

print(f'Database: {DB_PATH}')
print(f'Size: {os.path.getsize(DB_PATH) / 1024 / 1024:.1f} MB')
print()
print(f'{"Table":<25}  {"Rows":>10}')
print('-' * 38)
for t in tables['name']:
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {t}', conn).iloc[0]['n']
    print(f'{t:<25}  {n:>10,}')

conn.close()

print()
print('Park factors: in memory (30 parks, no DB needed)')
print()
print('Next steps:')
print('  • Run Cell 10 (nightly_update) each morning')
print('  • Change names in Cell 12 to explore any batter/pitcher matchup')
print('  • Phase 2: build the XGBoost prediction model (coming next)')